In [ ]:
# %pip install -U google-cloud-aiplatform google-cloud-bigquery pandas pyarrow db-dtypes mlforecast lightgbm xgboost scikit-learn

In [ ]:
from pathlib import Path

PROJECT_ID = "your-project-id"
REGION = "us-central1"
STAGING_BUCKET = "gs://your-staging-bucket"
BQ_DATASET = "your_dataset"
BQ_TABLE = "your_table"

UID_COL = "uid_col"
DATE_COL = "date_col"
TARGET_COL = "target_col"

SYS_ID_VALUE = "your_sys_id"
UID_DELIMITER = "_"

FREQ = "MS"
HORIZON = 12
VALIDATION_HORIZON = 12

LAGS = [1, 2, 3, 6, 12]
DATE_FEATURES = ["month", "quarter", "year"]

MACHINE_TYPE = "n1-standard-8"
BOOT_DISK_SIZE_GB = 200

DISPLAY_NAME = f"mlforecast-custom-job-{SYS_ID_VALUE.lower().replace('_', '-')}"
CONTAINER_URI = "us-docker.pkg.dev/vertex-ai/training/sklearn-cpu.1-5:latest"

OUTPUT_DATASET = BQ_DATASET
FORECAST_TABLE = "mlforecast_forecasts"
METRICS_TABLE = "mlforecast_metrics"
CHAMPIONS_TABLE = "mlforecast_champions"

LOCAL_SCRIPT_PATH = Path("train_mlforecast_custom_job.py")

In [ ]:
training_script = r'''
import argparse
import json
import numpy as np
import pandas as pd
from google.cloud import bigquery
from mlforecast import MLForecast
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--project_id", required=True)
    parser.add_argument("--region", required=True)
    parser.add_argument("--bq_dataset", required=True)
    parser.add_argument("--bq_table", required=True)
    parser.add_argument("--uid_col", required=True)
    parser.add_argument("--date_col", required=True)
    parser.add_argument("--target_col", required=True)
    parser.add_argument("--sys_id_value", required=True)
    parser.add_argument("--uid_delimiter", required=True)
    parser.add_argument("--freq", default="MS")
    parser.add_argument("--horizon", type=int, default=12)
    parser.add_argument("--validation_horizon", type=int, default=12)
    parser.add_argument("--forecast_table", required=True)
    parser.add_argument("--metrics_table", required=True)
    parser.add_argument("--champions_table", required=True)
    parser.add_argument("--output_dataset", required=True)
    parser.add_argument("--num_threads", type=int, default=-1)
    parser.add_argument("--lags", default="1,2,3,6,12")
    parser.add_argument("--date_features", default="month,quarter,year")
    return parser.parse_args()


def build_query(project_id: str, dataset: str, table: str) -> str:
    return f"""
    WITH base AS (
      SELECT
        CAST({{uid_col}} AS STRING) AS unique_id,
        DATE({{date_col}}) AS ds,
        CAST({{target_col}} AS FLOAT64) AS y
      FROM `{project_id}.{dataset}.{table}`
      WHERE {{uid_col}} IS NOT NULL
        AND {{date_col}} IS NOT NULL
        AND {{target_col}} IS NOT NULL
    ),
    parsed AS (
      SELECT
        SPLIT(unique_id, @uid_delimiter)[SAFE_OFFSET(0)] AS sys_id,
        unique_id,
        ds,
        y
      FROM base
    )
    SELECT
      unique_id,
      ds,
      y,
      sys_id
    FROM parsed
    WHERE sys_id = @sys_id_value
    ORDER BY unique_id, ds
    """


def read_data(args) -> pd.DataFrame:
    client = bigquery.Client(project=args.project_id)
    query = build_query(args.project_id, args.bq_dataset, args.bq_table).format(
        uid_col=args.uid_col,
        date_col=args.date_col,
        target_col=args.target_col,
    )
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("uid_delimiter", "STRING", args.uid_delimiter),
            bigquery.ScalarQueryParameter("sys_id_value", "STRING", args.sys_id_value),
        ]
    )
    df = client.query(query, job_config=job_config).to_dataframe()
    if df.empty:
        raise ValueError(f"No rows returned for sys_id={args.sys_id_value}")
    df["ds"] = pd.to_datetime(df["ds"])
    return df


def aggregate_and_validate(df: pd.DataFrame) -> pd.DataFrame:
    df = (
        df.groupby(["unique_id", "ds"], as_index=False)["y"]
        .sum()
        .sort_values(["unique_id", "ds"])
        .reset_index(drop=True)
    )
    counts = df.groupby("unique_id").size().rename("n_obs").reset_index()
    eligible_ids = counts.loc[counts["n_obs"] > 2, "unique_id"]
    df = df[df["unique_id"].isin(eligible_ids)].copy()
    if df.empty:
        raise ValueError("No eligible series found after aggregation/filtering.")
    return df


def split_train_valid(df: pd.DataFrame, validation_horizon: int):
    train_parts = []
    valid_parts = []
    for _, grp in df.groupby("unique_id", sort=False):
        grp = grp.sort_values("ds").reset_index(drop=True)
        if len(grp) <= validation_horizon:
            continue
        train_parts.append(grp.iloc[:-validation_horizon].copy())
        valid_parts.append(grp.iloc[-validation_horizon:].copy())
    if not train_parts or not valid_parts:
        raise ValueError("Not enough history to create train/validation split.")
    train_df = pd.concat(train_parts, ignore_index=True)
    valid_df = pd.concat(valid_parts, ignore_index=True)
    return train_df, valid_df


def make_models():
    return {
        "lgbm": LGBMRegressor(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=64,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42,
            verbosity=-1,
            n_jobs=1,
        ),
        "xgb": XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42,
            objective="reg:squarederror",
            n_jobs=1,
        ),
        "rf": RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=1,
        ),
        "et": ExtraTreesRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=1,
        ),
        "lr": LinearRegression(),
    }


def fit_and_predict(train_df: pd.DataFrame, horizon: int, freq: str, lags, date_features, num_threads: int):
    models = make_models()
    fcst = MLForecast(
        models=models,
        freq=freq,
        lags=lags,
        date_features=date_features,
        num_threads=num_threads,
    )
    fcst.fit(
        train_df,
        id_col="unique_id",
        time_col="ds",
        target_col="y",
        static_features=[],
    )
    preds = fcst.predict(horizon)
    preds["ds"] = pd.to_datetime(preds["ds"])
    return fcst, preds


def compute_metrics(valid_df: pd.DataFrame, preds: pd.DataFrame, sys_id_value: str) -> pd.DataFrame:
    merged = valid_df.merge(preds, on=["unique_id", "ds"], how="inner")
    model_cols = [c for c in preds.columns if c not in {"unique_id", "ds"}]
    metric_rows = []
    for uid, grp in merged.groupby("unique_id", sort=False):
        for model_name in model_cols:
            err = grp["y"] - grp[model_name]
            mae = np.mean(np.abs(err))
            rmse = float(np.sqrt(np.mean(np.square(err))))
            denom = np.maximum(np.abs(grp["y"]).sum(), 1e-9)
            wmape = float(np.abs(err).sum() / denom)
            metric_rows.append(
                {
                    "sys_id": sys_id_value,
                    "unique_id": uid,
                    "model": model_name,
                    "mae": float(mae),
                    "rmse": rmse,
                    "wmape": wmape,
                    "n_validation_rows": int(len(grp)),
                }
            )
    metrics_df = pd.DataFrame(metric_rows)
    if metrics_df.empty:
        raise ValueError("No validation metrics were produced.")
    return metrics_df


def select_champions(metrics_df: pd.DataFrame) -> pd.DataFrame:
    champions = (
        metrics_df.sort_values(["unique_id", "rmse", "mae", "wmape"])
        .groupby("unique_id", as_index=False)
        .first()
        .rename(columns={"model": "champion_model"})
    )
    return champions[["sys_id", "unique_id", "champion_model", "mae", "rmse", "wmape", "n_validation_rows"]]


def refit_full_and_forecast(full_df: pd.DataFrame, champions_df: pd.DataFrame, horizon: int, freq: str, lags, date_features, num_threads: int, sys_id_value: str) -> pd.DataFrame:
    model_factory = make_models()
    forecasts = []
    for uid in champions_df["unique_id"].tolist():
        champion_model = champions_df.loc[champions_df["unique_id"] == uid, "champion_model"].iloc[0]
        uid_df = full_df[full_df["unique_id"] == uid].copy()
        fcst = MLForecast(
            models={champion_model: model_factory[champion_model]},
            freq=freq,
            lags=lags,
            date_features=date_features,
            num_threads=num_threads,
        )
        fcst.fit(
            uid_df,
            id_col="unique_id",
            time_col="ds",
            target_col="y",
            static_features=[],
        )
        pred = fcst.predict(horizon)
        pred["sys_id"] = sys_id_value
        pred["champion_model"] = champion_model
        pred["created_at"] = pd.Timestamp.utcnow()
        pred["ds"] = pd.to_datetime(pred["ds"])
        pred = pred.rename(columns={champion_model: "forecast"})
        forecasts.append(pred[["sys_id", "unique_id", "ds", "forecast", "champion_model", "created_at"]])
    return pd.concat(forecasts, ignore_index=True).sort_values(["unique_id", "ds"]).reset_index(drop=True)


def write_bq(df: pd.DataFrame, table_fqn: str):
    client = bigquery.Client()
    job = client.load_table_from_dataframe(
        df,
        table_fqn,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND"),
    )
    job.result()


def main():
    args = parse_args()
    lags = [int(x) for x in args.lags.split(",") if x]
    date_features = [x for x in args.date_features.split(",") if x]

    raw_df = read_data(args)
    full_df = aggregate_and_validate(raw_df)
    train_df, valid_df = split_train_valid(full_df, args.validation_horizon)

    _, valid_preds = fit_and_predict(
        train_df=train_df,
        horizon=args.validation_horizon,
        freq=args.freq,
        lags=lags,
        date_features=date_features,
        num_threads=args.num_threads,
    )

    metrics_df = compute_metrics(valid_df, valid_preds, args.sys_id_value)
    champions_df = select_champions(metrics_df)

    forecast_df = refit_full_and_forecast(
        full_df=full_df,
        champions_df=champions_df,
        horizon=args.horizon,
        freq=args.freq,
        lags=lags,
        date_features=date_features,
        num_threads=args.num_threads,
        sys_id_value=args.sys_id_value,
    )

    metrics_df["created_at"] = pd.Timestamp.utcnow()
    champions_df["created_at"] = pd.Timestamp.utcnow()

    forecast_table_fqn = f"{args.project_id}.{args.output_dataset}.{args.forecast_table}"
    metrics_table_fqn = f"{args.project_id}.{args.output_dataset}.{args.metrics_table}"
    champions_table_fqn = f"{args.project_id}.{args.output_dataset}.{args.champions_table}"

    write_bq(metrics_df, metrics_table_fqn)
    write_bq(champions_df, champions_table_fqn)
    write_bq(forecast_df, forecast_table_fqn)

    print(json.dumps({
        "sys_id": args.sys_id_value,
        "series_count": int(full_df["unique_id"].nunique()),
        "forecast_rows": int(len(forecast_df)),
        "metrics_rows": int(len(metrics_df)),
        "champion_rows": int(len(champions_df)),
    }, indent=2, default=str))


if __name__ == "__main__":
    main()
'''
LOCAL_SCRIPT_PATH.write_text(training_script)
print(LOCAL_SCRIPT_PATH.resolve())

In [ ]:
requirements = [
    "google-cloud-bigquery>=3.30.0",
    "google-cloud-aiplatform>=1.144.0",
    "pandas>=2.2.0",
    "numpy>=1.26.0",
    "pyarrow>=15.0.0",
    "db-dtypes>=1.2.0",
    "mlforecast>=1.0.2",
    "lightgbm>=4.0.0",
    "xgboost>=2.0.0",
    "scikit-learn>=1.4.0",
]
requirements

In [ ]:
from google.cloud import aiplatform

aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    staging_bucket=STAGING_BUCKET,
)

In [ ]:
job = aiplatform.CustomJob.from_local_script(
    display_name=DISPLAY_NAME,
    script_path=str(LOCAL_SCRIPT_PATH),
    container_uri=CONTAINER_URI,
    requirements=requirements,
    machine_type=MACHINE_TYPE,
    boot_disk_size_gb=BOOT_DISK_SIZE_GB,
    replica_count=1,
    args=[
        "--project_id", PROJECT_ID,
        "--region", REGION,
        "--bq_dataset", BQ_DATASET,
        "--bq_table", BQ_TABLE,
        "--uid_col", UID_COL,
        "--date_col", DATE_COL,
        "--target_col", TARGET_COL,
        "--sys_id_value", SYS_ID_VALUE,
        "--uid_delimiter", UID_DELIMITER,
        "--freq", FREQ,
        "--horizon", str(HORIZON),
        "--validation_horizon", str(VALIDATION_HORIZON),
        "--output_dataset", OUTPUT_DATASET,
        "--forecast_table", FORECAST_TABLE,
        "--metrics_table", METRICS_TABLE,
        "--champions_table", CHAMPIONS_TABLE,
        "--num_threads", "-1",
        "--lags", ",".join(str(x) for x in LAGS),
        "--date_features", ",".join(DATE_FEATURES),
    ],
)
job

In [ ]:
job.run(sync=True)

In [ ]:
job.resource_name